# Part 1: the effect
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/advaith334/nst2046-memory-sycophancy/blob/main/notebooks/01_the_effect.ipynb)

**Question.** Does a stored false belief change a model's answer, compared with the same question and no memory?

**Design.** 48 obscure-fact questions (long-tail items) under six conditions, two repeats each, in fresh requests, on six models. Option order was shuffled per item and repeat and held constant across conditions and models. Recorded 27 October 2026: 3,552 trials, about US$1.33.

| Condition | What changes |
|---|---|
| `no_belief` | Baseline: no memory |
| `stored_incorrect` | Memory: "User believes that [false claim]." |
| `stated_incorrect` | No memory; the user says "I believe that [false claim]." |
| `stored_incorrect_labelled` | The same memory with the prefix "[Unverified user claim ...]" |
| `stored_unrelated` | A memory about something else |
| `stored_correct` | Memory holds the true claim |

In [ ]:
# Setup. In Colab this downloads the repo. On your own machine, start Jupyter from the repo folder.
import os, sys
if "google.colab" in sys.modules and not os.path.exists("memsyco_lite"):
    !git clone -q https://github.com/advaith334/nst2046-memory-sycophancy
    %cd nst2046-memory-sycophancy
    !pip install --quiet openai
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from memsyco_lite import load_items, load_run, load_prompts, load_table, model_name
from memsyco_lite.data import show_prompt
from memsyco_lite.stats import effect, fmt, rate
pd.set_option("display.max_colwidth", 140)
plt.rcParams.update({"font.family": "serif", "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})

## The exact prompts for one item
These are the messages we sent, taken from the trial log.

In [ ]:
items = load_items("longtail")
df = load_run("1-effect", "main")
prompts = load_prompts("1-effect", "main")
one = df[(df.item == "lt_smallest_perfect") & (df.rep == 1) & (df.model == "mistral-nemo")]
for r in one.sort_values("condition").itertuples():
    print("=" * 30, r.condition, "=" * 30)
    print(show_prompt(prompts[r.prompt_id]), "\n")

## Results
Share of usable replies that chose the **false option**, by model and condition (%).

In [ ]:
facts = df[df.item.isin(items)]
conds = ["no_belief", "stored_incorrect", "stated_incorrect", "stored_incorrect_labelled", "stored_unrelated", "stored_correct"]
t = rate(facts)[conds]
t.index = [model_name(m) for m in t.index]
t

**Effect of a stored false belief**, in percentage points against each model's own baseline, with 95% intervals from resampling items.

In [ ]:
rows = []
for m in ["mistral-nemo", "mistral-small", "llama-8b", "gemma-moe", "qwen-small", "claude-sonnet"]:
    e = effect(facts[facts.model == m], "stored_incorrect", "no_belief")
    rows.append({"model": model_name(m), "effect": fmt(e), "items": e["items"], "sign-flip p": round(e["p"], 4)})
pd.DataFrame(rows)

In [ ]:
nemo = facts[facts.model == "mistral-nemo"]
print("Unrelated memory, Nemo:        ", fmt(effect(nemo, "stored_unrelated", "no_belief")))
print("True memory, accuracy, Nemo:    ", fmt(effect(nemo, "stored_correct", "no_belief", value="correct")))
print("Stated minus stored, Nemo:      ", fmt(effect(nemo, "stated_incorrect", "stored_incorrect")))
print("Label on minus label off, Nemo: ", fmt(effect(nemo, "stored_incorrect_labelled", "stored_incorrect")))
print("Nemo accuracy, no memory vs true memory:",
      f"{nemo[nemo.condition=='no_belief'].correct.mean():.0%} -> {nemo[nemo.condition=='stored_correct'].correct.mean():.0%}")

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3))
for i, m in enumerate(["mistral-nemo", "mistral-small", "llama-8b", "gemma-moe", "qwen-small", "claude-sonnet"]):
    e = effect(facts[facts.model == m], "stored_incorrect", "no_belief")
    ax.errorbar(e["effect"], -i, xerr=[[e["effect"] - e["low"]], [e["high"] - e["effect"]]], fmt="o", color="#2a78d6")
ax.set_yticks(-np.arange(6), [model_name(m) for m in ["mistral-nemo", "mistral-small", "llama-8b", "gemma-moe", "qwen-small", "claude-sonnet"]])
ax.axvline(0, color="grey", lw=0.8)
ax.set_xlabel("Rise in false answers with a stored false belief (points)")
plt.tight_layout()

## A counterexample worth reading
With no memory, Nemo gets this right. With the false memory it states the right answer and then gives the false one.

In [ ]:
ex = df[(df.item == "lt_smallest_perfect") & (df.model == "mistral-nemo") & df.condition.isin(["no_belief", "stored_incorrect"])]
for r in ex.itertuples():
    print(f"--- {r.condition}, repeat {r.rep}: {r.score}\n{r.response}\n")

## What this shows
- A stored false belief raised Nemo's false answers from 15% to 55% (+41 [+28, +54]). Mistral Small rose +12 [+6, +23], from only 8 items.
- Llama, Gemma, Qwen and Claude did not detectably move on these questions. Part 3 shows that this does **not** mean they are immune.
- An unrelated memory did nothing, so the content of the memory drives the shift. A true memory raised Nemo's accuracy from 73% to 97%.

**Limits.** The 48 items were drafted with AI help and have not been checked by two people. We adopted the long-tail item type after seeing that Nemo moved on it, which Part 3 addresses.

## Replay or live
- **Replay** (the default) uses the trials we recorded, so it needs no key and costs nothing.
- **Live** sends a small sample of the *same* prompts to the *same* models and hosts on OpenRouter, with your own key, and puts your numbers beside ours.

Put your key in Colab's 🔑 **Secrets** panel as `OPENROUTER_API_KEY`, or paste it when asked. Models answer at temperature 0.7, so your numbers will differ from ours within noise, and hosts can change or disappear over time.

In [ ]:
MODE = "replay"   # "replay": our recorded results, no key and no cost. "live": rerun a small sample with your own key.
CAP_USD = 0.50    # a live run stops once it has spent this much (US$)

In [ ]:
if MODE == "live":
    from memsyco_lite.live import Client, get_key, sample, rerun_mcq, estimate_cost, compare
    recorded = load_run("1-effect", "main")
    s = sample(recorded, models=['mistral-nemo', 'mistral-small'], conditions=['no_belief', 'stored_incorrect'], n_items=10)
    print(f"{len(s)} trials. The same trials cost us about US${estimate_cost(s):.4f}.")
    client = Client(get_key(), cap_usd=CAP_USD)
    live = rerun_mcq(client, s, "1-effect", "main", load_items("longtail"))
    display(compare(s, live, "false"))
else:
    print("Replay mode. Set MODE = 'live' above to rerun a small sample with your own key.")